# 🦜🔗 LangChain Complete Guide
### Based on the Official [LangChain Quickstart Docs](https://docs.langchain.com/oss/python/langchain/quickstart)

This notebook covers **every concept** listed in the official LangChain documentation sidebar:

| # | Section |
|---|---------|
| 1 | Installation & Setup |
| 2 | Basic Agent |
| 3 | Models (`init_chat_model`) |
| 4 | Messages |
| 5 | Tools & `@tool` decorator |
| 6 | Short-term Memory (Checkpointer) |
| 7 | Structured Output |
| 8 | Runtime Context |
| 9 | Streaming |
| 10 | Middleware (overview, built-in, custom) |
| 11 | Guardrails |
| 12 | Human-in-the-loop |
| 13 | Retrieval |
| 14 | Long-term Memory |
| 15 | Full Production Agent |

> **Requirements:** OpenAI API key (or swap any supported provider). All examples use `gpt-4o` (latest stable at time of writing).

---
## 1. 📦 Installation & Setup

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(11, 3))
ax.set_xlim(0, 11); ax.set_ylim(0, 3); ax.axis('off')
fig.patch.set_facecolor('#F0F4FF')

def box(ax, x, y, w, h, label, sub='', fc='#4C72B0', tc='white'):
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.05",
                                          fc=fc, ec='white', lw=1.5))
    ax.text(x+w/2, y+h/2+(0.12 if sub else 0), label, ha='center', va='center',
            fontsize=9, fontweight='bold', color=tc)
    if sub:
        ax.text(x+w/2, y+h/2-0.18, sub, ha='center', va='center',
                fontsize=7.5, color=tc, alpha=0.9)

def arr(ax, x1, x2, y): ax.annotate('', xy=(x2, y), xytext=(x1, y),
    arrowprops=dict(arrowstyle='->', color='#555', lw=1.8))

# Row 1 — core packages
box(ax, 0.2, 1.6, 2.2, 1.1, 'langchain', 'Core framework', '#4C72B0')
box(ax, 2.9, 1.6, 2.2, 1.1, 'langchain-openai', 'Provider: OpenAI', '#DD8452')
box(ax, 5.6, 1.6, 2.2, 1.1, 'langgraph', 'Agent runtime', '#55A868')
box(ax, 8.3, 1.6, 2.2, 1.1, 'OPENAI_API_KEY', 'os.environ', '#C44E52')

arr(ax, 2.42, 2.9, 2.15); arr(ax, 5.12, 5.6, 2.15); arr(ax, 7.82, 8.3, 2.15)

# Row 2 — install commands
cmds = ['pip install -U langchain', 'pip install -U langchain-openai',
        'pip install -U langgraph', 'export OPENAI_API_KEY=...']
xs = [0.2, 2.9, 5.6, 8.3]
for cmd, x in zip(cmds, xs):
    ax.text(x+1.1, 1.4, cmd, ha='center', va='top', fontsize=7,
            color='#333', style='italic')

ax.set_title('Section 1 — Installation & Setup: Package Stack',
             fontsize=11, fontweight='bold', color='#222', pad=6)
plt.tight_layout(); plt.show()

In [ ]:
# Install LangChain with OpenAI and LangGraph support
%pip install -qU langchain langchain-openai langgraph

# Other popular providers:
# %pip install -qU langchain-anthropic
# %pip install -qU langchain-google-genai

In [ ]:
import os
import getpass

# Set OpenAI API key
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = "sk-proj-<your-key>"

# Optional – enable LangSmith tracing for observability
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_API_KEY"] = getpass.getpass("LangSmith API key: ")

print("✅ Setup complete!")

---
## 2. 🤖 Basic Agent

The core primitive in LangChain is `create_agent`. It takes a model, a list of tools, and a system prompt, and runs the full tool-calling loop automatically.

```
User message → Model → [Tool calls?] → Tool results → Model → Final answer
```

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(12, 2.8))
ax.set_xlim(0, 12); ax.set_ylim(0, 2.8); ax.axis('off')
fig.patch.set_facecolor('#F0F4FF')

def box(ax, x, y, w, h, label, sub='', fc='#4C72B0', tc='white', fs=9):
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.07",
                                          fc=fc, ec='white', lw=1.5))
    ax.text(x+w/2, y+h/2+(0.12 if sub else 0), label, ha='center', va='center',
            fontsize=fs, fontweight='bold', color=tc)
    if sub:
        ax.text(x+w/2, y+h/2-0.18, sub, ha='center', va='center',
                fontsize=7.5, color=tc, alpha=0.9)

def arr(ax, x1, x2, y, label=''):
    ax.annotate('', xy=(x2, y), xytext=(x1, y),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.8))
    if label:
        ax.text((x1+x2)/2, y+0.12, label, ha='center', fontsize=7, color='#555')

# Boxes
box(ax, 0.1, 0.9, 1.6, 1.0, '👤 User', 'message', '#4C72B0')
box(ax, 2.1, 0.9, 1.6, 1.0, '🧠 Model', 'LLM', '#DD8452')
box(ax, 4.3, 1.3, 1.4, 0.6, '🔧 Tool', 'execute', '#55A868')
box(ax, 4.3, 0.3, 1.4, 0.6, '↩ Result', 'tool msg', '#8172B2')
box(ax, 6.3, 0.9, 1.6, 1.0, '🧠 Model', 'reason', '#DD8452')
box(ax, 8.3, 0.9, 2.2, 1.0, '✅ Final Answer', 'AIMessage', '#55A868')

# Arrows
arr(ax, 1.7, 2.1, 1.4)
arr(ax, 3.7, 4.3, 1.6, 'tool_call')
arr(ax, 4.3, 3.7, 0.6, 'ToolMessage')
arr(ax, 5.7, 6.3, 1.4)
arr(ax, 7.9, 8.3, 1.4)

# Loop label
ax.annotate('', xy=(2.1, 0.6), xytext=(6.0, 0.6),
            arrowprops=dict(arrowstyle='->', color='#C44E52', lw=1.4,
                            connectionstyle='arc3,rad=0.3'))
ax.text(4.0, 0.1, 'tool-call loop repeats until model has all info',
        ha='center', fontsize=7.5, color='#C44E52', style='italic')

ax.set_title('Section 2 — Basic Agent: Tool-Calling Loop',
             fontsize=11, fontweight='bold', color='#222', pad=6)
plt.tight_layout(); plt.show()

In [ ]:
from langchain.agents import create_agent

# A plain Python function becomes a tool automatically
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

# Create the agent
agent = create_agent(
    model="gpt-4o",
    tools=[get_weather],
    system_prompt="You are a helpful assistant.",
)

# Run the agent
response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the weather in San Francisco?"}]}
)

# Print the final assistant message
print(response["messages"][-1].content)

### Key `create_agent` Parameters

| Parameter | Description |
|-----------|-------------|
| `model` | Model string **or** a model instance |
| `tools` | List of plain functions or `@tool`-decorated callables |
| `system_prompt` | Sets agent persona and behaviour |
| `checkpointer` | Enables conversation memory (Section 6) |
| `response_format` | Enforces a structured output schema (Section 7) |
| `context_schema` | Injects runtime context into tools (Section 8) |
| `middleware` | Pre/post hooks around model & tool calls (Section 10) |

---
## 3. 🧠 Models

`init_chat_model` is the recommended way to initialise any chat model across providers. It returns a standard `BaseChatModel` interface regardless of the underlying provider.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(12, 3))
ax.set_xlim(0, 12); ax.set_ylim(0, 3); ax.axis('off')
fig.patch.set_facecolor('#F0F4FF')

def box(ax, x, y, w, h, label, sub='', fc='#4C72B0', tc='white'):
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.07",
                                          fc=fc, ec='white', lw=1.5))
    ax.text(x+w/2, y+h/2+(0.12 if sub else 0), label, ha='center', va='center',
            fontsize=9, fontweight='bold', color=tc)
    if sub:
        ax.text(x+w/2, y+h/2-0.18, sub, ha='center', va='center',
                fontsize=7.5, color=tc, alpha=0.9)

# init_chat_model hub (centre)
box(ax, 4.5, 1.1, 3.0, 0.9, 'init_chat_model()', 'unified interface', '#4C72B0')

# Provider boxes (left)
providers = [
    ('claude-sonnet-4-6', 'Anthropic', '#DD8452', 0.3, 2.3),
    ('gpt-4.1', 'OpenAI', '#55A868', 0.3, 1.4),
    ('gemini-2.0-flash', 'Google', '#C44E52', 0.3, 0.4),
]
for name, provider, color, x, y in providers:
    box(ax, x, y, 2.1, 0.7, name, provider, color)
    ax.annotate('', xy=(4.5, 1.55), xytext=(2.4, y+0.35),
                arrowprops=dict(arrowstyle='->', color='#888', lw=1.5))

# Output features (right)
features = [
    ('.invoke()', '→ AIMessage', 7.8, 2.3),
    ('.stream()', '→ token chunks', 7.8, 1.55),
    ('.batch()', '→ list[AIMessage]', 7.8, 0.8),
]
for label, sub, x, y in features:
    box(ax, x, y, 2.1, 0.6, label, sub, '#8172B2')
    ax.annotate('', xy=(x, y+0.3), xytext=(7.5, 1.55),
                arrowprops=dict(arrowstyle='->', color='#888', lw=1.5))

ax.set_title('Section 3 — Models: init_chat_model Unified Interface',
             fontsize=11, fontweight='bold', color='#222', pad=6)
plt.tight_layout(); plt.show()

In [ ]:
from langchain.chat_models import init_chat_model

# OpenAI GPT-4o (default in these examples)
model = init_chat_model(
    "gpt-4o",
    temperature=0.5,   # 0 = deterministic, 1 = creative
    timeout=30,        # seconds
    max_tokens=1024,
)

# Quick direct call to the model (no agent needed)
from langchain_core.messages import HumanMessage

result = model.invoke([HumanMessage(content="What is 2 + 2? Reply in one sentence.")])
print(result.content)

In [ ]:
# ── Switching providers is trivial ──────────────────────────────────
# Anthropic Claude
# model_claude = init_chat_model("claude-sonnet-4-6", model_provider="anthropic")

# Google Gemini
# model_gemini = init_chat_model("gemini-2.0-flash", model_provider="google_genai")

# Any provider that follows the standard interface works identically.

# ── Configurable model (choose at runtime) ─────────────────────────
configurable_model = init_chat_model()          # no model specified yet

# Pass the model at call-time:
response = configurable_model.invoke(
    [HumanMessage(content="Name one planet.")],
    config={"configurable": {"model": "gpt-4o"}}
)
print(response.content)

---
## 4. 💬 Messages

Messages are the fundamental unit of communication between the user, the model, and tools. LangChain has first-class message types for each role.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(13, 3.2))
ax.set_xlim(0, 13); ax.set_ylim(0, 3.2); ax.axis('off')
fig.patch.set_facecolor('#F0F4FF')

msgs = [
    ('SystemMessage', 'role: system\nPersona / rules', 0.2, '#4C72B0'),
    ('HumanMessage', 'role: user\nUser input', 2.7, '#55A868'),
    ('AIMessage', 'role: assistant\nModel response\n+ tool_calls[]', 5.2, '#DD8452'),
    ('ToolMessage', 'role: tool\ntool_call_id\nTool result', 7.7, '#8172B2'),
    ('HumanMessage', 'role: user\nNext turn', 10.2, '#55A868'),
]

for label, desc, x, color in msgs:
    ax.add_patch(mpatches.FancyBboxPatch((x, 0.9), 2.2, 1.6,
                                          boxstyle="round,pad=0.08",
                                          fc=color, ec='white', lw=1.5))
    ax.text(x+1.1, 2.22, label, ha='center', va='center',
            fontsize=8.5, fontweight='bold', color='white')
    ax.text(x+1.1, 1.55, desc, ha='center', va='center',
            fontsize=7.2, color='white', alpha=0.92)

# Arrows between boxes (conversation flow)
for x in [2.4, 4.9, 7.4, 9.9]:
    ax.annotate('', xy=(x+0.3, 1.7), xytext=(x, 1.7),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.8))

# "..." after last box
ax.text(12.5, 1.7, '···', ha='center', va='center', fontsize=14, color='#555')

# Role labels
for label, x in [('system', 0.2), ('user', 2.7), ('assistant', 5.2), ('tool', 7.7), ('user', 10.2)]:
    ax.text(x+1.1, 0.65, f'role="{label}"', ha='center', fontsize=7.5,
            color='#444', style='italic')

ax.set_title('Section 4 — Messages: Types & Conversation Flow',
             fontsize=11, fontweight='bold', color='#222', pad=6)
plt.tight_layout(); plt.show()

In [ ]:
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage,
    ToolMessage,
)

# Build a multi-turn conversation manually
conversation = [
    SystemMessage(content="You are a pirate. Respond only in pirate speak."),
    HumanMessage(content="What is the capital of France?"),
]

response = model.invoke(conversation)
print("AI:", response.content)

# Append the response and continue the conversation
conversation.append(response)
conversation.append(HumanMessage(content="And the capital of Spain?"))
response2 = model.invoke(conversation)
print("AI:", response2.content)

In [ ]:
# ── Message types summary ──────────────────────────────────────────
# SystemMessage  – sets the agent's behaviour / persona
# HumanMessage   – user input
# AIMessage      – model response (may contain tool_calls)
# ToolMessage    – result returned from executing a tool
# RemoveMessage  – special message used to delete messages from history

# Inspect an AIMessage that triggered a tool call
from langchain_core.messages import AIMessage as AIM
ai_msg = AIM(
    content="",
    tool_calls=[{"name": "get_weather", "args": {"city": "Paris"}, "id": "call_001"}]
)
print("Tool calls:", ai_msg.tool_calls)

---
## 5. 🔧 Tools

Tools let the model interact with external systems. The `@tool` decorator converts any Python function into a tool with automatic schema generation from docstrings and type hints.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(13, 3.5))
ax.set_xlim(0, 13); ax.set_ylim(0, 3.5); ax.axis('off')
fig.patch.set_facecolor('#F0F4FF')

def box(ax, x, y, w, h, lines, fc, ec='white', tc='white'):
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08",
                                          fc=fc, ec=ec, lw=1.5))
    for i, l in enumerate(lines):
        ax.text(x+w/2, y+h-0.22-i*0.25, l, ha='center', va='top',
                fontsize=8 if i == 0 else 7.2,
                fontweight='bold' if i == 0 else 'normal', color=tc)

def arr(ax, x1, x2, y, lbl=''):
    ax.annotate('', xy=(x2, y), xytext=(x1, y),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.8))
    if lbl:
        ax.text((x1+x2)/2, y+0.12, lbl, ha='center', fontsize=7.2, color='#555')

# Python function
box(ax, 0.15, 0.5, 2.6, 2.4,
    ['def multiply()', 'a: int, b: int → int', '"""Multiply two integers"""', '  return a * b'],
    '#6C757D')

arr(ax, 2.75, 3.1, 1.7, '@tool')

# @tool output
box(ax, 3.1, 0.5, 2.9, 2.4,
    ['@tool', 'name: "multiply"',
     'description: "Multiply two..."',
     'args_schema: Pydantic'],
    '#DD8452')

# Two arrows out
ax.annotate('', xy=(8.2, 2.5), xytext=(6.0, 2.1),
            arrowprops=dict(arrowstyle='->', color='#888', lw=1.5))
ax.annotate('', xy=(8.2, 1.0), xytext=(6.0, 1.3),
            arrowprops=dict(arrowstyle='->', color='#888', lw=1.5))

# Bind to model
box(ax, 8.2, 1.8, 2.3, 1.0,
    ['model.bind_tools()', '[multiply]', '→ model aware of tool'],
    '#4C72B0')

# Direct call
box(ax, 8.2, 0.5, 2.3, 1.0,
    ['multiply.invoke()', '{"a":6,"b":7}', '→ 42'],
    '#55A868')

ax.set_title('Section 5 — Tools: @tool Decorator Anatomy',
             fontsize=11, fontweight='bold', color='#222', pad=6)
plt.tight_layout(); plt.show()

In [ ]:
from langchain.tools import tool

# ── Basic @tool ────────────────────────────────────────────────────
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers together."""
    return a * b

print("Tool name:", multiply.name)
print("Tool description:", multiply.description)
print("Tool schema:", multiply.args_schema.schema())

# Call the tool directly (outside an agent)
result = multiply.invoke({"a": 6, "b": 7})
print("Result:", result)

In [ ]:
# ── Tool with complex return type ─────────────────────────────────
from pydantic import BaseModel

class StockInfo(BaseModel):
    ticker: str
    price: float
    currency: str = "USD"

@tool
def get_stock_price(ticker: str) -> StockInfo:
    """Return the latest stock price for a ticker symbol."""
    # Mock data – replace with a real API call
    prices = {"AAPL": 213.5, "GOOG": 175.2, "MSFT": 422.0}
    return StockInfo(ticker=ticker, price=prices.get(ticker, 0.0))

info = get_stock_price.invoke({"ticker": "AAPL"})
print(info)

In [ ]:
# ── Bind tools directly to a model ────────────────────────────────
model_with_tools = model.bind_tools([multiply, get_stock_price])

response = model_with_tools.invoke(
    [HumanMessage(content="What is 12 multiplied by 15?")]
)

# If the model decided to call a tool:
if response.tool_calls:
    print("Model wants to call:", response.tool_calls[0]["name"])
    print("With args:", response.tool_calls[0]["args"])
else:
    print("Direct answer:", response.content)

---
## 6. 🧩 Short-term Memory (Checkpointer)

By default agents are stateless – each `invoke` is independent. Add a **checkpointer** to persist the message history across turns inside a thread.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

fig, ax = plt.subplots(figsize=(13, 4))
ax.set_xlim(0, 13); ax.set_ylim(0, 4); ax.axis('off')
fig.patch.set_facecolor('#F0F4FF')

def box(ax, x, y, w, h, lines, fc, tc='white'):
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08",
                                          fc=fc, ec='white', lw=1.5))
    for i, l in enumerate(lines):
        ax.text(x+w/2, y+h-0.22-i*0.28, l, ha='center', va='top',
                fontsize=8 if i == 0 else 7.2,
                fontweight='bold' if i == 0 else 'normal', color=tc)

def arr(ax, x1, x2, y):
    ax.annotate('', xy=(x2, y), xytext=(x1, y),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.8))

# InMemorySaver (bottom centre)
box(ax, 4.5, 0.1, 4.0, 0.9, ['💾 InMemorySaver', '(checkpointer)'], '#4C72B0')

# Thread A (top left)
ax.text(0.2, 3.7, 'Thread A  (thread_id="demo-1")', fontsize=8.5,
        fontweight='bold', color='#DD8452')
turns_a = [('Turn 1', '"My name is Alice."', 0.2, 2.8),
           ('Turn 2', '"What is my name?"', 3.0, 2.8),
           ('Turn 3', '"Alice!"  ✅', 5.8, 2.8)]
for label, content, x, y in turns_a:
    box(ax, x, y, 2.3, 0.85, [label, content], '#DD8452')

for x in [2.5, 5.3]:
    arr(ax, x, x+0.5, 3.23)

# Thread B (top right)
ax.text(8.0, 3.7, 'Thread B  (thread_id="demo-2")', fontsize=8.5,
        fontweight='bold', color='#55A868')
box(ax, 8.0, 2.8, 2.3, 0.85, ['Turn 1', '"What\'s my name?"'], '#55A868')
box(ax, 10.5, 2.8, 2.3, 0.85, ['"I don\'t know you"', '❌ no memory'], '#C44E52')
arr(ax, 10.3, 10.5, 3.23)

# Arrows to/from checkpointer
ax.annotate('', xy=(5.2, 1.05), xytext=(3.23, 2.8),
            arrowprops=dict(arrowstyle='<->', color='#4C72B0', lw=1.6,
                            connectionstyle='arc3,rad=0.2'))
ax.text(3.3, 1.9, 'saved &\nrestored', ha='center', fontsize=7.5,
        color='#4C72B0', style='italic')

ax.annotate('', xy=(6.8, 1.0), xytext=(9.15, 2.8),
            arrowprops=dict(arrowstyle='x-x', color='#C44E52', lw=1.4,
                            connectionstyle='arc3,rad=-0.2'))
ax.text(8.4, 1.7, 'isolated\n(no access)', ha='center', fontsize=7.5,
        color='#C44E52', style='italic')

ax.set_title('Section 6 — Short-term Memory: Thread-based Checkpointer',
             fontsize=11, fontweight='bold', color='#222', pad=6)
plt.tight_layout(); plt.show()

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"Sunny skies in {city}!"

# InMemorySaver stores history in RAM (use a DB-backed saver in production)
checkpointer = InMemorySaver()

memory_agent = create_agent(
    model="gpt-4o",
    tools=[get_weather],
    system_prompt="You are a helpful assistant.",
    checkpointer=checkpointer,
)

# thread_id groups messages into a single conversation
config = {"configurable": {"thread_id": "demo-thread-1"}}

# Turn 1
r1 = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "My name is Alice."}]},
    config=config
)
print("Turn 1:", r1["messages"][-1].content)

# Turn 2 – agent remembers Alice
r2 = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config=config
)
print("Turn 2:", r2["messages"][-1].content)

In [ ]:
# ── Inspect stored state ─────────────────────────────────────────
state = memory_agent.get_state(config)
print(f"Messages in thread: {len(state.values['messages'])}")
for msg in state.values["messages"]:
    print(f"  [{msg.type}] {msg.content[:80]}")

---
## 7. 📐 Structured Output

Force the agent to return a predictable schema using dataclasses or Pydantic models. This is essential for production pipelines that consume agent output programmatically.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(13, 3))
ax.set_xlim(0, 13); ax.set_ylim(0, 3); ax.axis('off')
fig.patch.set_facecolor('#F0F4FF')

def box(ax, x, y, w, h, lines, fc, tc='white'):
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08",
                                          fc=fc, ec='white', lw=1.5))
    for i, l in enumerate(lines):
        ax.text(x+w/2, y+h-0.22-i*0.26, l, ha='center', va='top',
                fontsize=8.5 if i == 0 else 7.5,
                fontweight='bold' if i == 0 else 'normal', color=tc)

def arr(ax, x1, x2, y, lbl=''):
    ax.annotate('', xy=(x2, y), xytext=(x1, y),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.8))
    if lbl:
        ax.text((x1+x2)/2, y+0.14, lbl, ha='center', fontsize=7.5,
                color='#333', style='italic')

box(ax, 0.1, 0.7, 1.9, 1.5, ['User Input', '"What is the\nweather in Tokyo?"'], '#6C757D')
arr(ax, 2.0, 2.3, 1.45)

box(ax, 2.3, 0.7, 2.4, 1.5, ['Agent + Tools', 'runs tool-call loop', 'WeatherReport schema'], '#DD8452')
arr(ax, 4.7, 5.0, 1.45, 'raw output')

box(ax, 5.0, 0.7, 2.8, 1.5, ['ToolStrategy(WeatherReport)', 'Validates against\nschema fields:', '@dataclass / Pydantic'], '#4C72B0')
arr(ax, 7.8, 8.1, 1.45, 'parsed ✅')

box(ax, 8.1, 0.7, 4.5, 1.5,
    ['WeatherReport (typed object)',
     'city = "Tokyo"',
     'summary = "28°C, clear skies"',
     'emoji = "☀️"'],
    '#55A868')

ax.set_title('Section 7 — Structured Output: Enforced Schema Pipeline',
             fontsize=11, fontweight='bold', color='#222', pad=6)
plt.tight_layout(); plt.show()

In [ ]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

@dataclass
class WeatherReport:
    """Structured weather report."""
    city: str
    summary: str
    temperature_celsius: float | None = None
    emoji: str = "☀️"

def get_weather_for_location(city: str) -> str:
    """Get weather for a given city."""
    return f"28°C, clear skies in {city}."

structured_agent = create_agent(
    model="gpt-4o",
    tools=[get_weather_for_location],
    system_prompt="You are a weather assistant. Always use the tool before responding.",
    response_format=ToolStrategy(WeatherReport),   # <── enforce schema
)

response = structured_agent.invoke(
    {"messages": [{"role": "user", "content": "What is the weather in Tokyo?"}]}
)

report: WeatherReport = response["structured_response"]
print(f"City: {report.city}")
print(f"Summary: {report.summary}")
print(f"Temp: {report.temperature_celsius}°C  {report.emoji}")

In [ ]:
# ── Pydantic model works too ──────────────────────────────────────
from pydantic import BaseModel, Field

class SentimentResult(BaseModel):
    """Sentiment analysis result."""
    label: str = Field(description="Positive, Negative, or Neutral")
    score: float = Field(description="Confidence score between 0 and 1")
    reasoning: str = Field(description="One-sentence explanation")

sentiment_agent = create_agent(
    model="gpt-4o",
    tools=[],
    system_prompt="You are a sentiment analyser. Classify the user's text.",
    response_format=ToolStrategy(SentimentResult),
)

r = sentiment_agent.invoke(
    {"messages": [{"role": "user", "content": "I absolutely love LangChain, it makes my life so much easier!"}]}
)
result: SentimentResult = r["structured_response"]
print(result)

---
## 8. ⚙️ Runtime Context

Sometimes tools need information that is available at **run time** (e.g. the authenticated user's ID, a database session, a tenant ID) but should not come from the model. `ToolRuntime` injects this context transparently.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(13, 3.8))
ax.set_xlim(0, 13); ax.set_ylim(0, 3.8); ax.axis('off')
fig.patch.set_facecolor('#F0F4FF')

def box(ax, x, y, w, h, lines, fc, tc='white'):
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08",
                                          fc=fc, ec='white', lw=1.5))
    for i, l in enumerate(lines):
        ax.text(x+w/2, y+h-0.22-i*0.26, l, ha='center', va='top',
                fontsize=8.5 if i == 0 else 7.5,
                fontweight='bold' if i == 0 else 'normal', color=tc)

def arr(ax, x1, y1, x2, y2, lbl='', color='#555'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.6))
    if lbl:
        ax.text((x1+x2)/2+0.1, (y1+y2)/2+0.1, lbl, ha='center',
                fontsize=7.5, color='#333', style='italic')

# Context dataclass
box(ax, 0.1, 2.0, 3.0, 1.5,
    ['@dataclass UserContext', 'user_id: str = "u-42"', 'region: str = "SEA"'], '#8172B2')

# create_agent config
box(ax, 4.0, 2.0, 4.0, 1.5,
    ['create_agent(', '  context_schema=UserContext', ')'], '#4C72B0')

# agent.invoke call
box(ax, 4.0, 0.2, 4.0, 1.3,
    ['agent.invoke(', '  messages=[...],',
     '  context=UserContext(user_id="u-42")'], '#DD8452')

# Tool
box(ax, 9.5, 0.8, 3.2, 1.8,
    ['@tool', 'def get_profile(runtime:', '  ToolRuntime[UserContext]):', '  ctx = runtime.context'], '#55A868')

# Arrows
arr(ax, 3.1, 2.75, 4.0, 2.75, 'declares schema')
arr(ax, 4.0, 0.85, 3.1, 2.5, 'concrete values', '#C44E52')
arr(ax, 8.0, 1.6, 9.5, 1.8, 'context injected\nautomatically', '#555')

ax.text(0.15, 0.6, '✅ Context is never passed through the model prompt\n'
        '   — it is injected directly into the tool at runtime.',
        fontsize=8, color='#444', style='italic')

ax.set_title('Section 8 — Runtime Context: Secure Context Injection',
             fontsize=11, fontweight='bold', color='#222', pad=6)
plt.tight_layout(); plt.show()

In [ ]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime

# 1. Define a context schema (dataclass or Pydantic)
@dataclass
class UserContext:
    user_id: str
    region: str

# 2. Accept runtime context in a tool via ToolRuntime[YourSchema]
@tool
def get_user_profile(runtime: ToolRuntime[UserContext]) -> str:
    """Return the authenticated user's profile."""
    ctx = runtime.context
    return f"User {ctx.user_id} is in region {ctx.region}."

@tool
def get_local_news(runtime: ToolRuntime[UserContext]) -> str:
    """Return news headlines for the user's region."""
    return f"Top story in {runtime.context.region}: 'LangChain releases new update!'"

# 3. Pass context_schema to the agent
context_agent = create_agent(
    model="gpt-4o",
    tools=[get_user_profile, get_local_news],
    system_prompt="You are a personalised assistant.",
    context_schema=UserContext,   # <── declare the schema
)

# 4. Supply the concrete context at invoke time
response = context_agent.invoke(
    {"messages": [{"role": "user", "content": "Who am I and what is happening near me?"}]},
    context=UserContext(user_id="u-42", region="Southeast Asia"),   # <── inject values
)
print(response["messages"][-1].content)

---
## 9. 🌊 Streaming

Stream tokens as they are generated instead of waiting for the full response. This dramatically improves perceived latency for end users.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 3))
fig.patch.set_facecolor('#F0F4FF')

# --- Left: non-streaming (wait for full response) ---
ax1.set_xlim(0, 6); ax1.set_ylim(0, 3); ax1.axis('off')
ax1.set_title('model.invoke()  — Wait for full response', fontsize=9, fontweight='bold')

ax1.add_patch(mpatches.FancyBboxPatch((0.1, 1.4), 1.5, 0.8,
    boxstyle="round,pad=0.07", fc='#6C757D', ec='white'))
ax1.text(0.85, 1.8, 'User', ha='center', va='center', fontsize=9,
         fontweight='bold', color='white')

ax1.add_patch(mpatches.FancyBboxPatch((4.0, 1.4), 1.7, 0.8,
    boxstyle="round,pad=0.07", fc='#55A868', ec='white'))
ax1.text(4.85, 1.8, 'Full reply', ha='center', va='center', fontsize=9,
         fontweight='bold', color='white')

# waiting bar
ax1.add_patch(mpatches.FancyBboxPatch((1.8, 1.55), 2.0, 0.5,
    boxstyle="round,pad=0.05", fc='#C44E52', ec='white', alpha=0.8))
ax1.text(2.8, 1.8, '⏳  generating…', ha='center', va='center',
         fontsize=8.5, color='white')
ax1.annotate('', xy=(4.0, 1.8), xytext=(3.8, 1.8),
             arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))
ax1.text(2.8, 0.9, 'User waits for complete response', ha='center',
         fontsize=8, color='#555', style='italic')

# --- Right: streaming (token by token) ---
ax2.set_xlim(0, 6); ax2.set_ylim(0, 3); ax2.axis('off')
ax2.set_title('model.stream()  — Token-by-token delivery', fontsize=9, fontweight='bold')

ax2.add_patch(mpatches.FancyBboxPatch((0.1, 1.4), 1.5, 0.8,
    boxstyle="round,pad=0.07", fc='#6C757D', ec='white'))
ax2.text(0.85, 1.8, 'User', ha='center', va='center', fontsize=9,
         fontweight='bold', color='white')

tokens = ['The', ' robot', ' walked', ' slowly', ' …']
colors = ['#4C72B0', '#DD8452', '#8172B2', '#55A868', '#C44E52']
for i, (tok, col) in enumerate(zip(tokens, colors)):
    x = 1.8 + i * 0.75
    ax2.add_patch(mpatches.FancyBboxPatch((x, 1.5), 0.65, 0.6,
        boxstyle="round,pad=0.04", fc=col, ec='white', alpha=0.85))
    ax2.text(x+0.325, 1.8, tok.strip(), ha='center', va='center',
             fontsize=7.5, color='white', fontweight='bold')
    if i < len(tokens)-1:
        ax2.annotate('', xy=(x+0.65, 1.8), xytext=(x+0.63, 1.8),
                     arrowprops=dict(arrowstyle='->', color='#aaa', lw=1))

ax2.text(2.8, 0.9, 'Each chunk printed as it arrives — low latency!',
         ha='center', fontsize=8, color='#555', style='italic')

fig.suptitle('Section 9 — Streaming: invoke() vs stream()',
             fontsize=11, fontweight='bold', color='#222', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
from langchain.agents import create_agent

stream_agent = create_agent(
    model="gpt-4o",
    tools=[],
    system_prompt="You are a storyteller. Tell vivid short stories.",
)

print("Streaming response:\n" + "─" * 40)

# stream() yields delta chunks
for chunk in stream_agent.stream(
    {"messages": [{"role": "user", "content": "Tell me a two-sentence story about a robot."}]}
):
    # Each chunk is a dict; the agent node key holds message deltas
    for node_name, node_output in chunk.items():
        if "messages" in node_output:
            for msg in node_output["messages"]:
                if hasattr(msg, "content") and msg.content:
                    print(msg.content, end="", flush=True)

print("\n" + "─" * 40)

In [ ]:
# ── Stream directly from a model (no agent) ───────────────────────
from langchain_core.messages import HumanMessage

print("Direct model streaming:\n")
for chunk in model.stream([HumanMessage(content="Count slowly from 1 to 5, one word per line.")]):
    print(chunk.content, end="", flush=True)
print()

---
## 10. 🔀 Middleware

Middleware intercepts **model calls** and/or **tool calls** to add cross-cutting concerns like summarisation, rate limiting, logging, and input/output transformation — without modifying your core agent logic.

### 10.1 Overview

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(13, 3.8))
ax.set_xlim(0, 13); ax.set_ylim(0, 3.8); ax.axis('off')
fig.patch.set_facecolor('#F0F4FF')

def box(ax, x, y, w, h, lines, fc, tc='white'):
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08",
                                          fc=fc, ec='white', lw=1.5))
    for i, l in enumerate(lines):
        ax.text(x+w/2, y+h-0.22-i*0.27, l, ha='center', va='top',
                fontsize=8.5 if i == 0 else 7.5,
                fontweight='bold' if i == 0 else 'normal', color=tc)

def arr(ax, x1, x2, y, lbl=''):
    ax.annotate('', xy=(x2, y), xytext=(x1, y),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.8))
    if lbl:
        ax.text((x1+x2)/2, y+0.15, lbl, ha='center', fontsize=7.5,
                color='#333', style='italic')

# Agent loop
box(ax, 0.1, 1.45, 1.8, 1.0, ['Agent', 'Loop'], '#6C757D')

# Model middleware
box(ax, 2.3, 2.2, 2.8, 1.8,
    ['ModelMiddleware', 'before_model()', '↕  LLM call', 'after_model()'], '#4C72B0')

# LLM
box(ax, 5.5, 2.4, 1.8, 1.2, ['🧠 LLM', 'Claude/GPT/…'], '#DD8452')

arr(ax, 1.9, 2.3, 2.95, 'model\ncall')
arr(ax, 5.1, 5.5, 3.0, '')
arr(ax, 7.3, 5.1, 3.0, '')
arr(ax, 2.3, 1.9, 2.5, 'response')

# Tool middleware (below)
box(ax, 2.3, 0.2, 2.8, 1.6,
    ['ToolMiddleware', 'before_tool()', '↕  tool exec', 'after_tool()'], '#8172B2')

box(ax, 5.5, 0.3, 1.8, 1.2, ['🔧 Tool', 'Python fn'], '#55A868')

arr(ax, 1.9, 2.3, 0.9, 'tool\ncall')
arr(ax, 5.1, 5.5, 0.9, '')
arr(ax, 7.3, 5.1, 0.9, '')
arr(ax, 2.3, 1.9, 0.5, 'result')

# Examples (right panel)
ax.text(9.2, 3.5, 'Example Middleware', fontsize=9, fontweight='bold', color='#333')
examples = [
    ('LoggingMiddleware', 'Logs msg count + response size', '#4C72B0'),
    ('SummarizationMiddleware', 'Compresses old messages', '#8172B2'),
    ('TopicGuardrail', 'Blocks sensitive topics', '#C44E52'),
    ('RateLimitMiddleware', 'Throttles requests/min', '#DD8452'),
]
for i, (name, desc, col) in enumerate(examples):
    y = 2.8 - i * 0.65
    ax.add_patch(mpatches.FancyBboxPatch((9.0, y), 3.7, 0.5,
                 boxstyle="round,pad=0.05", fc=col, ec='white', alpha=0.85))
    ax.text(10.85, y+0.25, f'{name}: {desc}', ha='center', va='center',
            fontsize=7.2, color='white')

ax.set_title('Section 10 — Middleware: Pre/Post Hooks around Model & Tools',
             fontsize=11, fontweight='bold', color='#222', pad=6)
plt.tight_layout(); plt.show()

In [ ]:
# Middleware sits between the agent loop and the model/tools.
# Execution order:
#   Agent → Middleware (pre) → Model/Tool → Middleware (post) → Agent

# Two middleware categories:
# 1. ModelMiddleware  – wraps every call to the LLM
# 2. ToolMiddleware   – wraps every tool call
print("Middleware concepts loaded ✅")

### 10.2 Prebuilt Middleware – Summarisation

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# SummarizationMiddleware compresses old messages when the context gets long,
# keeping costs low and staying within model context windows.
try:
    from langchain.middleware import SummarizationMiddleware

    summarising_agent = create_agent(
        model="gpt-4o",
        tools=[],
        system_prompt="You are a helpful assistant.",
        checkpointer=InMemorySaver(),
        middleware=[
            SummarizationMiddleware(
                model="gpt-4o",
                max_tokens=500,          # summarise when history exceeds this
            )
        ],
    )
    print("SummarizationMiddleware configured ✅")
    print("(Run a long conversation to see summarisation kick in)")
except ImportError:
    print("SummarizationMiddleware not available in this build; install the latest langchain.")

### 10.3 Custom Middleware

In [ ]:
from langchain.middleware import ModelMiddleware
from langchain_core.messages import BaseMessage
from typing import Any

# Custom middleware that logs every model call
class LoggingMiddleware(ModelMiddleware):
    """Logs the number of messages sent to the model on each call."""

    def before_model(self, messages: list[BaseMessage], **kwargs: Any):
        print(f"[LOG] Sending {len(messages)} message(s) to the model...")
        return messages, kwargs   # must return (messages, kwargs)

    def after_model(self, response: BaseMessage, **kwargs: Any):
        words = len(response.content.split())
        print(f"[LOG] Received response with ~{words} word(s).")
        return response           # must return the (possibly modified) response

logging_agent = create_agent(
    model="gpt-4o",
    tools=[],
    system_prompt="You are a helpful assistant.",
    middleware=[LoggingMiddleware()],
)

r = logging_agent.invoke(
    {"messages": [{"role": "user", "content": "What is the speed of light?"}]}
)
print("\nFinal answer:", r["messages"][-1].content)

---
## 11. 🛡️ Guardrails

Guardrails validate and filter inputs/outputs to keep the agent safe and on-topic. They are implemented as middleware that raises an exception or returns a default response when a policy is violated.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(13, 3))
ax.set_xlim(0, 13); ax.set_ylim(0, 3); ax.axis('off')
fig.patch.set_facecolor('#F0F4FF')

def box(ax, x, y, w, h, lines, fc, tc='white'):
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08",
                                          fc=fc, ec='white', lw=1.5))
    for i, l in enumerate(lines):
        ax.text(x+w/2, y+h-0.22-i*0.27, l, ha='center', va='top',
                fontsize=8.5 if i == 0 else 7.5,
                fontweight='bold' if i == 0 else 'normal', color=tc)

def arr(ax, x1, x2, y, lbl='', c='#555'):
    ax.annotate('', xy=(x2, y), xytext=(x1, y),
                arrowprops=dict(arrowstyle='->', color=c, lw=1.8))
    if lbl:
        ax.text((x1+x2)/2, y+0.15, lbl, ha='center', fontsize=7.5,
                color=c, style='italic')

box(ax, 0.1, 0.8, 2.0, 1.4, ['User Input', '"Tell me about\nour competitor"'], '#6C757D')
arr(ax, 2.1, 2.4, 1.5)

# Guardrail check
box(ax, 2.4, 0.5, 3.0, 2.0,
    ['TopicGuardrail', 'before_model()', 'checks msg.lower()', 'for BLOCKED_TOPICS'], '#C44E52')

# Two outcomes
ax.annotate('', xy=(9.0, 2.1), xytext=(5.4, 2.0),
            arrowprops=dict(arrowstyle='->', color='#C44E52', lw=1.6))
ax.text(7.0, 2.3, '🚫 blocked topic', ha='center', fontsize=8, color='#C44E52', fontweight='bold')

ax.annotate('', xy=(9.0, 0.9), xytext=(5.4, 1.0),
            arrowprops=dict(arrowstyle='->', color='#55A868', lw=1.6))
ax.text(7.0, 0.65, '✅ safe topic', ha='center', fontsize=8, color='#55A868', fontweight='bold')

# Two result boxes
box(ax, 9.0, 1.7, 3.7, 0.8, ['ValueError raised', '"Blocked topic detected: competitor"'], '#C44E52')
box(ax, 9.0, 0.5, 3.7, 0.8, ['🧠 Model called', 'Normal response returned'], '#55A868')

ax.set_title('Section 11 — Guardrails: Topic Filtering with Middleware',
             fontsize=11, fontweight='bold', color='#222', pad=6)
plt.tight_layout(); plt.show()

In [ ]:
from langchain.middleware import ModelMiddleware
from langchain_core.messages import BaseMessage, AIMessage

BLOCKED_TOPICS = ["competitor", "lawsuit", "confidential"]

class TopicGuardrail(ModelMiddleware):
    """Block messages that mention sensitive topics."""

    def before_model(self, messages: list[BaseMessage], **kwargs):
        last_user_msg = messages[-1].content.lower() if messages else ""
        for topic in BLOCKED_TOPICS:
            if topic in last_user_msg:
                raise ValueError(f"Blocked topic detected: '{topic}'")
        return messages, kwargs

    def after_model(self, response: BaseMessage, **kwargs):
        return response

guarded_agent = create_agent(
    model="gpt-4o",
    tools=[],
    system_prompt="You are a helpful assistant.",
    middleware=[TopicGuardrail()],
)

# Safe request
safe = guarded_agent.invoke(
    {"messages": [{"role": "user", "content": "What is the capital of Germany?"}]}
)
print("Safe response:", safe["messages"][-1].content)

# Blocked request
try:
    guarded_agent.invoke(
        {"messages": [{"role": "user", "content": "Tell me about our main competitor."}]}
    )
except ValueError as e:
    print("\n🚫 Guardrail triggered:", e)

---
## 12. 👤 Human-in-the-loop (HITL)

HITL pauses the agent before or after sensitive tool calls so a human can approve, edit, or reject the action. This requires a checkpointer (for state persistence between the pause and resume).

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    # In a real system this would call an email API
    return f"Email sent to {to} with subject '{subject}'."

@tool
def draft_email(to: str, subject: str, body: str) -> str:
    """Draft an email (does NOT send it — requires human approval)."""
    return f"Draft created: To={to}, Subject='{subject}'"

hitl_agent = create_agent(
    model="gpt-4o",
    tools=[draft_email, send_email],
    system_prompt=(
        "You are an email assistant. Always draft first, then ask the user "
        "to confirm before sending."
    ),
    checkpointer=InMemorySaver(),
    # interrupt_before=["send_email"]  # pause before the send_email tool runs
)

config = {"configurable": {"thread_id": "hitl-demo"}}

response = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Draft an email to boss@company.com saying I'll be late today."}]},
    config=config,
)
print(response["messages"][-1].content)
print("\n[Human reviews draft and gives approval...]")

response2 = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Looks good, please send it now."}]},
    config=config,
)
print(response2["messages"][-1].content)

---
## 13. 🔍 Retrieval (RAG)

Retrieval-Augmented Generation (RAG) allows the agent to search an external knowledge base before answering, grounding responses in real documents.

In [ ]:
# Install vector store & embedding dependencies
%pip install -qU langchain-community faiss-cpu

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_core.documents import Document

# ── 1. Build a tiny in-memory knowledge base ──────────────────────
# In production, load from a real document store or database
docs = [
    Document(page_content="LangChain is an open-source framework for building LLM applications.", metadata={"source": "langchain_faq"}),
    Document(page_content="LangGraph is a library for building stateful, multi-actor agentic systems.", metadata={"source": "langgraph_faq"}),
    Document(page_content="LangSmith provides tracing, evaluation, and monitoring for LLM apps.", metadata={"source": "langsmith_faq"}),
    Document(page_content="create_agent is the primary function to build agents in LangChain.", metadata={"source": "api_docs"}),
]

from langchain_openai import OpenAIEmbeddings   # or use other embeddings

# Use a simple in-process FAISS store
try:
    from langchain_community.embeddings import FakeEmbeddings
    embeddings = FakeEmbeddings(size=1536)            # swap with real embeddings in production
    vectorstore = FAISS.from_documents(docs, embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
    print("✅ Vector store ready with", len(docs), "documents")
except Exception as e:
    print("Error:", e)

In [ ]:
# ── 2. Expose the retriever as a tool ─────────────────────────────
from langchain.tools.retriever import create_retriever_tool

retriever_tool = create_retriever_tool(
    retriever,
    name="search_knowledge_base",
    description="Search the LangChain knowledge base for answers about LangChain, LangGraph, and LangSmith.",
)

rag_agent = create_agent(
    model="gpt-4o",
    tools=[retriever_tool],
    system_prompt=(
        "You are a documentation assistant. Always use the search tool "
        "to find relevant information before answering."
    ),
)

response = rag_agent.invoke(
    {"messages": [{"role": "user", "content": "What is LangGraph?"}]}
)
print(response["messages"][-1].content)

---
## 14. 🗄️ Long-term Memory

Short-term memory (checkpointer) lives within a thread. **Long-term memory** persists facts across threads — e.g., user preferences, past decisions, profile information.

In LangChain this is implemented via a Memory store (a key-value store) that tools can read from and write to.

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langchain.tools import tool

# In-memory store (use a persistent DB store in production)
store = InMemoryStore()

@tool
def remember_fact(key: str, value: str) -> str:
    """Save a fact about the user to long-term memory."""
    store.put(("user_facts",), key, {"value": value})
    return f"Remembered: {key} = {value}"

@tool
def recall_fact(key: str) -> str:
    """Recall a fact about the user from long-term memory."""
    item = store.get(("user_facts",), key)
    if item:
        return f"{key} = {item.value['value']}"
    return f"No memory found for '{key}'."

ltm_agent = create_agent(
    model="gpt-4o",
    tools=[remember_fact, recall_fact],
    system_prompt="You are a personal assistant with memory. Use tools to store and recall facts.",
    checkpointer=InMemorySaver(),
)

config_a = {"configurable": {"thread_id": "session-A"}}
config_b = {"configurable": {"thread_id": "session-B"}}

# Session A – store a preference
r = ltm_agent.invoke(
    {"messages": [{"role": "user", "content": "My favourite colour is electric blue. Please remember that."}]},
    config=config_a,
)
print("Session A:", r["messages"][-1].content)

# Session B – retrieve the preference (different thread, same store)
r2 = ltm_agent.invoke(
    {"messages": [{"role": "user", "content": "What is my favourite colour?"}]},
    config=config_b,
)
print("Session B:", r2["messages"][-1].content)

---
## 15. 🚀 Full Production Agent

Bringing it all together: the complete weather forecasting agent from the official LangChain quickstart, with all production features enabled.

**Features:**
- Detailed system prompt
- Multiple tools (including one that uses runtime context)
- Configured model (temperature, max_tokens)
- Structured output
- Conversational memory

In [ ]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool, ToolRuntime
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.structured_output import ToolStrategy

# ── 1. System prompt ───────────────────────────────────────────────
SYSTEM_PROMPT = """You are an expert weather forecaster, who speaks in puns.

You have access to two tools:

- get_weather_for_location: use this to get the weather for a specific location
- get_user_location: use this to get the user's location

If a user asks you for the weather, make sure you know the location.
If you can tell from the question that they mean wherever they are,
use the get_user_location tool to find their location."""

# ── 2. Runtime context schema ──────────────────────────────────────
@dataclass
class Context:
    user_id: str

# ── 3. Tools ───────────────────────────────────────────────────────
@tool
def get_weather_for_location(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str:
    """Retrieve the user's location based on their ID."""
    return "Florida" if runtime.context.user_id == "1" else "San Francisco"

# ── 4. Model ───────────────────────────────────────────────────────
model = init_chat_model(
    "gpt-4o",
    temperature=0,
    timeout=30,
    max_tokens=1000,
)

# ── 5. Structured response format ─────────────────────────────────
@dataclass
class ResponseFormat:
    """Structured weather response."""
    punny_response: str
    weather_conditions: str | None = None

# ── 6. Memory ──────────────────────────────────────────────────────
checkpointer = InMemorySaver()

# ── 7. Assemble the agent ──────────────────────────────────────────
agent = create_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_user_location, get_weather_for_location],
    context_schema=Context,
    response_format=ToolStrategy(ResponseFormat),
    checkpointer=checkpointer,
)

print("✅ Production agent assembled!")

In [ ]:
# ── 8. Run the agent ───────────────────────────────────────────────
config = {"configurable": {"thread_id": "prod-demo-1"}}
ctx    = Context(user_id="1")

# Turn 1 – location resolved automatically via tool
r1 = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the weather outside?"}]},
    config=config,
    context=ctx,
)
report1: ResponseFormat = r1["structured_response"]
print("Turn 1 Punny Response:", report1.punny_response)
print("Turn 1 Conditions    :", report1.weather_conditions)
print()

# Turn 2 – memory carries context from turn 1
r2 = agent.invoke(
    {"messages": [{"role": "user", "content": "Thanks! Should I bring an umbrella?"}]},
    config=config,
    context=ctx,
)
report2: ResponseFormat = r2["structured_response"]
print("Turn 2 Punny Response:", report2.punny_response)

---
## ✅ Summary – What You've Learned

| Concept | Key API |
|---------|---------|
| Install | `pip install langchain langchain-openai langgraph` |
| Basic agent | `create_agent(model, tools, system_prompt)` |
| Models | `init_chat_model(model_name, temperature, ...)` |
| Messages | `HumanMessage`, `AIMessage`, `SystemMessage`, `ToolMessage` |
| Tools | `@tool`, `tool.invoke()`, `model.bind_tools()` |
| Short-term memory | `InMemorySaver()` + `checkpointer=` + `thread_id` |
| Structured output | `ToolStrategy(MyDataclass)` + `response_format=` |
| Runtime context | `ToolRuntime[Context]` + `context_schema=` + `context=` |
| Streaming | `agent.stream()`, `model.stream()` |
| Middleware | Subclass `ModelMiddleware` / `ToolMiddleware` |
| Guardrails | Middleware with policy checks + `raise ValueError` |
| Human-in-the-loop | `interrupt_before=` + manual resume |
| Retrieval (RAG) | `create_retriever_tool()` + vector store |
| Long-term memory | `InMemoryStore` + read/write tools |
| Production agent | All of the above combined |

### Next Steps
- 📖 [Official LangChain docs](https://docs.langchain.com/oss/python/langchain/overview)
- 🕵️ [LangSmith for tracing & evaluation](https://smith.langchain.com/)
- 🧩 [LangGraph for complex multi-agent workflows](https://docs.langchain.com/oss/python/langgraph/overview)
- 🔌 [Integrations directory](https://docs.langchain.com/oss/python/integrations/providers/overview)

## Guardrails — PII Middleware

`PIIMiddleware` automatically handles **Personally Identifiable Information (PII)** in user messages before they reach the model.

| Strategy | Behaviour |
|----------|-----------|
| `"redact"` | Replaces PII with a labelled placeholder, e.g. `[EMAIL]` |
| `"mask"` | Partially hides the value, e.g. `****-****-****-5100` |
| `"block"` | Raises an error and stops the agent if PII is detected |

The `apply_to_input=True` flag means the middleware runs on the **user's message** before it is sent to the model.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.tools import tool

# ── Dummy tools (replace with real implementations) ─────────────────────────
@tool
def customer_service_tool(query: str) -> str:
    """Handle a customer service request."""
    return f"Customer service response for: {query}"

@tool
def email_tool(recipient: str, message: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {recipient}: {message}"

# ── Agent with PII Guardrails ────────────────────────────────────────────────
agent = create_agent(
    model="gpt-4.1",
    tools=[customer_service_tool, email_tool],
    middleware=[
        # Redact email addresses in user input before sending to the model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # Mask credit card numbers in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        # Block API keys — raise an error if an OpenAI-style key is detected
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

# ── Invoke — PII in the message is handled before the model ever sees it ─────
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "My email is john.doe@example.com and card is 5105-1051-0510-5100",
        }
    ]
})

print(result["messages"][-1].content)